# Phase 257 f8nt: classification geometry control at the 7,513-chain pool, CPU: fold 8, arm `no-topo`, seeds 0 and 1

Continuation of Phase 255 (folds 0-3 finished on GPU; Kaggle's 30 h weekly GPU quota then ran out). Same data, folds (KFold seed 1337), model,
loss, early stopping and seeds as Phase 255; only the device differs. The two seeds run as two separate 2-thread processes so both use the
4 cores. Primary comparison across the ten folds: `geom+topo` vs `geom` on test PR-AUC.

In [ ]:
!pip install -q torch-geometric==2.6.1

In [ ]:
WORKER_SRC = 'import json, os, time, warnings\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch_geometric.loader import DataLoader\nfrom sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score\nfrom sklearn.model_selection import KFold\nwarnings.filterwarnings("ignore")\n\ndef locate(f):\n    c=[]\n    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):\n        if r.exists(): c.extend(r.rglob(f))\n    if not c: raise FileNotFoundError(f)\n    return max(c, key=lambda p: p.stat().st_size)\n\nblob = torch.load(locate("phase52_graphs.pt"), map_location="cpu", weights_only=False)\nN_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])\nassert N_PRE == 507 and TOPO_DIM == 34, (N_PRE, TOPO_DIM)\nfeats = {}\nfor t in "abc":\n    g = torch.load(locate(f"phase250{t}_geometry_scale3.pt"), map_location="cpu", weights_only=False)\n    feats.update(g["feats"]); print(f"geometry chunk {t}: {g[\'n_chains\']}/{g[\'n_total\']} accepted")\nGEOM_DIM = 17; SIDE_DIM = TOPO_DIM + GEOM_DIM\n\ndataset, dropped = [], 0\nfor d in blob["dataset"]:                                       # blob order, restricted to chains with geometry (identical in every kernel)\n    if (str(d.pdb_id), str(d.chain_id)) in feats: dataset.append(d)\n    else: dropped += 1\ndel blob\nassert dropped <= 0.02 * (len(dataset) + dropped), f"too many chains without geometry: {dropped}"\nallg = np.concatenate([feats[(str(d.pdb_id), str(d.chain_id))] for d in dataset], 0)\ng_mu = torch.from_numpy(allg.mean(0).astype(np.float32)); g_sd = torch.from_numpy(np.maximum(allg.std(0), 1e-6).astype(np.float32)); del allg\nfor d in dataset:\n    f = feats[(str(d.pdb_id), str(d.chain_id))]; assert f.shape == (d.x.size(0), GEOM_DIM)\n    d.x = torch.cat([d.x, ((torch.from_numpy(f) - g_mu) / g_sd).clamp(-5, 5)], dim=1)\ndel feats\nprint(f"chains kept {len(dataset):,}, dropped for missing geometry {dropped}; node dim {dataset[0].x.size(1)} = {N_PRE} + {TOPO_DIM} + {GEOM_DIM}")\n\nGPU_OK=False\nif torch.cuda.is_available():\n    try:\n        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True\n        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")\n    except RuntimeError as e: print("GPU unusable:", e)\nDEVICE = torch.device("cuda" if GPU_OK else "cpu")\nif hasattr(dataset[0], "cluster"):\n    assert len({d.cluster for d in dataset}) == len(dataset), "not 1 chain per cluster -- KFold would leak homologs"\n    print("verified: one chain per cluster, KFold over chains is homology-clean")\nelse:\n    print("WARNING: no `cluster` attribute on the graphs; homology-cleanliness of the folds is not re-verified here")\n\nclass EGCL(nn.Module):\n    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):\n        super().__init__(); s.uc=uc\n        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())\n        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))\n        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())\n        if uc:\n            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))\n            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)\n        s.norm=nn.LayerNorm(h)\n    def forward(s,h,x,ei,ea):\n        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)\n        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)\n        agg=torch.zeros_like(h).index_add_(0,dst,m)\n        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)\n        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))\n        if s.uc:\n            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg\n        return h,x\nclass Fus(nn.Module):\n    def __init__(s,h,t):\n        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())\n    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)\nclass Net(nn.Module):\n    def __init__(s,n_pre,td,h=128,L=4,dr=0.1):\n        super().__init__(); s.n_pre=n_pre\n        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))\n        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])\n        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])\n        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))\n    def forward(s,d):\n        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]\n        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)\n        return s.head(h).squeeze(-1)\n\ndef run_epoch(model, loader, opt=None):\n    tr = opt is not None; model.train(tr)\n    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))\n    ps,ys=[],[]\n    for b in loader:\n        b=b.to(DEVICE)\n        with torch.set_grad_enabled(tr):\n            lg=model(b); loss=crit(lg,b.y)\n        if tr:\n            opt.zero_grad(set_to_none=True); loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()\n        ps.append(torch.sigmoid(lg).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())\n    p,y=np.concatenate(ps),np.concatenate(ys)\n    return {"pr":average_precision_score(y,p),"roc":roc_auc_score(y,p),"p":p,"y":y}\n\ndef best_mcc(p,y):\n    return max(matthews_corrcoef(y,(p>=t).astype(int))\n               for t in np.linspace(0.05,0.95,37) if (p>=t).sum())\n\n# ---- one seed of one (fold, arm), run as its own process ----------------------------------------------------------\nimport sys, resource\nSEED = int(sys.argv[1]); FOLD_TO_RUN, ARM_TO_RUN = 8, "no-topo"\ntorch.set_num_threads(2)\nprint("worker seed", SEED, "| device", DEVICE, "| threads", torch.get_num_threads(), "| RSS MB", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss // 1024, flush=True)\n\nN_FOLDS, EPOCHS, PATIENCE, BATCH = 10, 40, 8, 16\nKEEP_SIDE = {"no-topo": [], "full": list(range(TOPO_DIM)), "geom": list(range(TOPO_DIM, SIDE_DIM)), "geom+topo": list(range(SIDE_DIM))}\nm_ = torch.zeros(SIDE_DIM); m_[KEEP_SIDE[ARM_TO_RUN]] = 1.0\nfor d in dataset: d.x[:, N_PRE:] = d.x[:, N_PRE:] * m_          # in place: unselected side columns zeroed (identical result to Phase 255\'s apply_arm)\nfolds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=1337).split(np.arange(len(dataset))))\ntr_idx, te_idx = folds[FOLD_TO_RUN]\nt0 = time.time()\nrng = np.random.RandomState(500 + SEED)\nperm = rng.permutation(tr_idx)\nnv = max(1, int(0.15 * len(perm)))\nva_ds = [dataset[i] for i in perm[:nv]]; tr_ds = [dataset[i] for i in perm[nv:]]; te_ds = [dataset[i] for i in te_idx]\ntorch.manual_seed(SEED); np.random.seed(SEED)\nm = Net(N_PRE, SIDE_DIM).to(DEVICE)\nopt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)\nsch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)\ntl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True); vl = DataLoader(va_ds, batch_size=BATCH)\nbest, state, bad = -1, None, 0\nfor ep in range(EPOCHS):\n    run_epoch(m, tl, opt); v = run_epoch(m, vl); sch.step()\n    print(f"    seed {SEED} epoch {ep}: val PR {v[\'pr\']:.3f} [{(time.time()-t0)/60:.0f}m]", flush=True)\n    if v["pr"] > best: best, state, bad = v["pr"], {k: t.clone() for k, t in m.state_dict().items()}, 0\n    else:\n        bad += 1\n        if bad >= PATIENCE: break\nm.load_state_dict(state)\nte = run_epoch(m, DataLoader(te_ds, batch_size=BATCH))\nrec = {"arm": ARM_TO_RUN, "fold": FOLD_TO_RUN, "seed": SEED, "val_pr": float(best), "test_pr": float(te["pr"]),\n       "test_roc": float(te["roc"]), "n_test": int(len(te["y"])), "n_pos": int(te["y"].sum()), "epochs": ep + 1}\njson.dump(rec, open(f"/kaggle/working/rec_seed{SEED}.json", "w"), indent=2, default=float)\nprint(f"DONE seed {SEED}: test PR {te[\'pr\']:.3f} ROC {te[\'roc\']:.3f} [{(time.time()-t0)/60:.0f}m] peak RSS MB {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss // 1024}", flush=True)\n'
open('/kaggle/working/worker.py', 'w').write(WORKER_SRC)
print('worker script written,', len(WORKER_SRC), 'chars')

In [ ]:
import subprocess, sys, os, time, json
procs = []
for seed in (0, 1):
    env = dict(os.environ, OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
    log = open(f"/kaggle/working/worker_seed{seed}.log", "w")
    procs.append((seed, subprocess.Popen([sys.executable, "-u", "/kaggle/working/worker.py", str(seed)], stdout=log, stderr=subprocess.STDOUT, env=env), log))
t0 = time.time()
while any(p.poll() is None for _, p, _ in procs):
    time.sleep(600)
    for seed, p, _ in procs:
        lines = open(f"/kaggle/working/worker_seed{seed}.log").read().strip().split("\n")
        print(f"[{(time.time()-t0)/60:.0f}m] seed {seed} (rc={p.poll()}): {lines[-1][:160]}", flush=True)
for seed, p, log in procs:
    log.close(); print(f"--- seed {seed} exit code {p.returncode}; last log lines:"); print("\n".join(open(f"/kaggle/working/worker_seed{seed}.log").read().strip().split("\n")[-4:]))
assert all(p.returncode == 0 for _, p, _ in procs), "a worker failed -- see its log above"
recs = [json.load(open(f"/kaggle/working/rec_seed{s}.json")) for s in (0, 1)]
json.dump({"records": recs, "folds": [recs[0]["fold"]]}, open("/kaggle/working/phase257_f8nt_records.json", "w"), indent=2)
print("saved phase257_f8nt_records.json:", [(r["seed"], round(r["test_pr"], 4)) for r in recs])